# UNI2-h Training on WILDS CAMELYON17 (Colab, Resume-Safe)

Trains `scripts/train_uni2h_preprocessed_colab.py` on **preprocessed WILDS H5** (`train_x/y.h5`, `valid_x/y.h5`) — same Macenko layout as the Virchow WILDS run. Checkpoints go to Google Drive (`RUN_DIR`).

**Before first run**
1. **Runtime → Change runtime type → GPU**
2. Accept the gated license at [MahmoodLab/UNI2-h](https://huggingface.co/MahmoodLab/UNI2-h)
3. Paste your Hugging Face **read token** in the config cell below (`HF_TOKEN`). Do **not** commit tokens to GitHub. Do **not** use `huggingface-cli login` — set the token in the config cell only.
4. Run the **Pre-download UNI2-h** cell once per Colab session (avoids Hub rate limits during training).

UNI2-h: frozen ViT-H/14-reg8, **1536-d** CLS embedding. WILDS batch size **64** (lower if OOM).

**Deterministic protocol (no MC dropout):** `--head-dropout 0 --mc-samples 0` in the training cell.

If `valid_*` was built with `--valid-source val`, validation is **OOD** (hospital holdout).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
import os
if os.path.isdir('GP_ECG'):
    %cd GP_ECG
    !git pull
else:
    !git clone https://github.com/LeenRayess/GP_ECG.git GP_ECG
    %cd GP_ECG

## Config — edit paths and `HF_TOKEN`

If your repo lives on Drive instead of GitHub, skip the clone cell and set `REPO_DIR` to that folder.

In [ ]:
REPO_DIR = '/content/GP_ECG'
PREPROCESSED_DIR = '/content/drive/MyDrive/GP_ECG_DATA/wilds/camelyon17_h5_full_oodval/preprocessed_macenko_benchmark_style'
RUN_DIR = '/content/drive/MyDrive/GP_ECG_RUNS/uni2h_wilds_preprocessed_run_01'
HF_TOKEN = ''  # paste from https://huggingface.co/settings/tokens

import os
os.makedirs(RUN_DIR, exist_ok=True)
if not HF_TOKEN:
    raise ValueError('Set HF_TOKEN in this cell before continuing.')
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN

In [ ]:
import os
import shutil

# Copy preprocessed .h5 from Drive → local SSD (faster reads). RUN_DIR stays on Drive.
LOCAL_PREPROCESSED_DIR = '/content/local_wilds_preprocessed_h5'

_src = PREPROCESSED_DIR
if _src.startswith('/content/drive/'):
    if not os.path.isdir(_src):
        raise FileNotFoundError('Drive path not found — check PREPROCESSED_DIR:\n  ' + _src)
    if not os.path.exists(LOCAL_PREPROCESSED_DIR):
        print('Copying preprocessed data Drive → local SSD (may take several minutes)...')
        print('  from:', _src)
        shutil.copytree(_src, LOCAL_PREPROCESSED_DIR)
        print('  done:', LOCAL_PREPROCESSED_DIR)
    else:
        print('Using existing local copy:', LOCAL_PREPROCESSED_DIR)
    PREPROCESSED_DIR = LOCAL_PREPROCESSED_DIR
else:
    print('PREPROCESSED_DIR is not under /content/drive/ — using as-is:', _src)

## Pre-download UNI2-h weights (run once per Colab session)

Caches the model under `/root/.cache/huggingface/`. Skip if you already ran this cell successfully in this session.

In [ ]:
import os
from huggingface_hub import snapshot_download

path = snapshot_download(
    repo_id='MahmoodLab/UNI2-h',
    token=os.environ['HF_TOKEN'],
)
print('Cached at:', path)

In [ ]:
%cd {REPO_DIR}
!python -m pip install --upgrade pip -q
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q timm h5py tqdm huggingface_hub scikit-learn

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA not available — switch to a GPU runtime before training.')

In [ ]:
%cd {REPO_DIR}
!python scripts/train_uni2h_preprocessed_colab.py \
  --preprocessed-dir "{PREPROCESSED_DIR}" \
  --out-dir "{RUN_DIR}" \
  --epochs 10 \
  --batch-size 64 \
  --num-workers 2 \
  --head-dropout 0 \
  --mc-samples 0 \
  --resume \
  --save-every-epoch-copy

In [ ]:
import json
import os

print('Run dir:', RUN_DIR)
artifacts = [
    'checkpoint_last.pt', 'model_best.pt', 'metrics_history.json', 'metrics_final.json',
    'metrics_final_detailed.json', 'temperature_fit.json', 'run_config.json', 'run_manifest.json',
    'run_progress.json', 'val_predictions.npz',
]
for name in artifacts:
    p = os.path.join(RUN_DIR, name)
    print(('OK ' if os.path.exists(p) else 'MISSING '), p)

hist = os.path.join(RUN_DIR, 'metrics_history.json')
if os.path.exists(hist):
    with open(hist, 'r', encoding='utf-8') as f:
        rows = json.load(f)
    if rows:
        print('Last epoch record:', rows[-1])

det = os.path.join(RUN_DIR, 'metrics_final_detailed.json')
if os.path.exists(det):
    with open(det, 'r', encoding='utf-8') as f:
        d = json.load(f)
    print('metrics_final_detailed keys:', list(d.keys()))